## Greenery Data Exploration

Layers explored & built:
1. DBSN `ar_vrd`   → parks / gardens (types `01`, `95`)
2. DBSN `pe_unis`  → uniform open spaces (parks, gardens, playgrounds)
3. OSM             → `leisure=park` / `garden`
4. Municipality data source

Output: `{city}_greenery.gpkg` with layers `osm_public` and/or `municipality`.
This file is the input for `300_walkability_isochrone_computation.ipynb`.

In [ ]:
# ── CHANGE THIS TO SWITCH CITY ──────────────────────────────────────────────
CITY = "bologna"
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import fiona
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import osmnx as ox
import pandas as pd

os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")

from config import CRS_WGS, CRS_METRIC, get_city_cfg, get_output_paths

cfg   = get_city_cfg(CITY)
src   = cfg["data_sources"]
paths = get_output_paths(CITY)

# Output .gpkg for this city — consumed by isochrone notebook
# For cities with one park source the key matches; for multi-source (genova, torino)
# you save separate files and list both keys in cfg["park_layers"].
OUT_GPKG = src[cfg["park_layers"][0]]   # primary output path

print(f"City   : {CITY}")
print(f"Output : {OUT_GPKG}")

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def area_ha(gdf):
    gdf = gdf.copy()
    gdf["area_ha"] = gdf.geometry.area / 1e4
    return gdf

def clip_and_reset(gdf, boundary):
    return gdf.pipe(gpd.clip, boundary).reset_index(drop=True)

def describe_areas(gdf):
    print(f"  Features : {len(gdf):,}")
    if "area_ha" in gdf.columns:
        d = gdf["area_ha"].describe().round(2)
        print(f"  Area (ha): median={d['50%']}, total={gdf['area_ha'].sum():.1f}")

## 0. Boundary

In [ ]:
boundary    = gpd.read_file(src["boundary"]).to_crs(CRS_METRIC)
boundary_ha = boundary.geometry.area.sum() / 1e4
boundary_wgs84 = boundary.to_crs(CRS_WGS).geometry.union_all()
print(f"Boundary: {boundary_ha:.0f} ha")

## 1. DBSN — ar_vrd (area verde)
Only runs if a DBSN geodatabase is available for this city. Skip otherwise.

In [ ]:
# Set DBSN_GDB to the path of the city's DBSN .gdb file, or None to skip.
DBSN_GDB = None   # e.g. "data/BO_dbsn_2025-07-28/Bologna_.gdb"

if DBSN_GDB:
    ar_vrd = gpd.read_file(DBSN_GDB, layer="ar_vrd").to_crs(CRS_METRIC)
    print(f"Total features: {len(ar_vrd):,}")
    print(ar_vrd["ar_vrd_ty"].value_counts().to_string())

    av_green_areas = (
        ar_vrd
        .query('ar_vrd_ty in ["01", "95"]')
        .pipe(clip_and_reset, boundary)
        .pipe(area_ha)
    )
    av_green_areas = av_green_areas[av_green_areas["area_ha"] > 0.5]
    print("ar_vrd parks+gardens (types 01,95), > 0.5 ha:")
    describe_areas(av_green_areas)
else:
    print("DBSN_GDB not set — skipping ar_vrd")
    av_green_areas = None

## 2. DBSN — pe_unis (settlement units)
Types: 11=park/garden, 1001=playground, 1101=garden, 1103=park

In [ ]:
if DBSN_GDB:
    pe_raw = gpd.read_file(DBSN_GDB, layer="pe_uins").to_crs(CRS_METRIC)
    print(f"Total features: {len(pe_raw):,}")

    GREEN_TYPES = ["11", "1001", "1101", "1102", "1103"]
    pe_green = (
        pe_raw
        .pipe(clip_and_reset, boundary)
        .pipe(area_ha)
        .loc[lambda df: df["pe_uins_ty"].isin(GREEN_TYPES)]
    )
    print("pe_unis green types:")
    describe_areas(pe_green)
else:
    print("DBSN_GDB not set — skipping pe_unis")
    pe_green = None

## 3. OSM — leisure = park / garden

In [ ]:
PRIVATE_ACCESS = {"private", "no", "restricted", "permit", "customers"}

osm_raw = (
    ox.features_from_polygon(boundary_wgs84, tags={"leisure": ["park", "garden"]})
    .to_crs(CRS_METRIC)
    .pipe(clip_and_reset, boundary)
    .pipe(area_ha)
)
osm_raw = osm_raw[osm_raw.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

osm_public = osm_raw.loc[
    osm_raw["access"].isna() | ~osm_raw["access"].isin(PRIVATE_ACCESS)
].copy()

print(f"OSM parks+gardens — all: {len(osm_raw)}")
print(f"OSM parks+gardens — public:")
describe_areas(osm_public)

## 4. Municipality data source
Only runs if the city config has a `parks_municipal` or `parks_dsnb` key pointing to a local file.

In [ ]:
# Attempt to load a municipality-provided parks file if configured
municipal_key = next(
    (k for k in ["parks_municipal", "parks_dsnb"] if k in src),
    None
)

if municipal_key:
    # Check if this file already exists (i.e. this run is an update, not first-time)
    import os as _os
    fpath = src[municipal_key]
    if _os.path.exists(fpath):
        muni_raw = (
            gpd.read_file(fpath)
            .to_crs(CRS_METRIC)
            .pipe(clip_and_reset, boundary)
            .pipe(area_ha)
        )
        print(f"Municipality parks ({municipal_key}):")
        describe_areas(muni_raw)
    else:
        print(f"File not found: {fpath} — skipping municipality layer")
        muni_raw = None
else:
    print("No municipality parks configured for this city")
    muni_raw = None

## 5. Summary comparison

In [ ]:
rows = [{"layer": "OSM public", "n": len(osm_public), "total_ha": osm_public.area_ha.sum(),
         "pct": osm_public.area_ha.sum() / boundary_ha * 100}]

if muni_raw is not None:
    rows.append({"layer": "Municipality", "n": len(muni_raw), "total_ha": muni_raw.area_ha.sum(),
                 "pct": muni_raw.area_ha.sum() / boundary_ha * 100})

if pe_green is not None:
    rows.append({"layer": "DBSN pe_unis", "n": len(pe_green), "total_ha": pe_green.area_ha.sum(),
                 "pct": pe_green.area_ha.sum() / boundary_ha * 100})

summary = pd.DataFrame(rows).round(1)
print(summary.to_string(index=False))

## 6. Source choice visualisation
Visual comparison to inform which layers to save for the isochrone pipeline.

In [ ]:
C_OSM   = "#00e676"
C_MUNI  = "#FFD54F"
C_SETT  = "#c44dff"
FIG_BG  = "#1a1a2e"
PAN_BG  = "#12122a"
LAND_BG = "#2e2e4a"

n_panels = 1 + (muni_raw is not None) + (pe_green is not None)
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 7), dpi=150)
if n_panels == 1:
    axes = [axes]
fig.patch.set_facecolor(FIG_BG)
for ax in axes:
    ax.set_facecolor(PAN_BG)
    boundary.plot(ax=ax, color=LAND_BG, zorder=0)
    boundary.boundary.plot(ax=ax, color="white", linewidth=0.8, alpha=0.5, zorder=5)
    ax.axis("off")

idx = 0
osm_public.plot(ax=axes[idx], color=C_OSM, alpha=0.7, zorder=3)
axes[idx].set_title(f"OSM public ({len(osm_public)})", color="white", fontsize=12)
idx += 1

if muni_raw is not None:
    muni_raw.plot(ax=axes[idx], color=C_MUNI, alpha=0.7, zorder=3)
    axes[idx].set_title(f"Municipality ({len(muni_raw)})", color="white", fontsize=12)
    idx += 1

if pe_green is not None:
    pe_green.plot(ax=axes[idx], color=C_SETT, alpha=0.7, zorder=3)
    axes[idx].set_title(f"DBSN pe_unis ({len(pe_green)})", color="white", fontsize=12)

fig.suptitle(f"{CITY.title()} — green source comparison", color="white", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Save — edit layers_to_save based on visual inspection above

In [ ]:
def prep_save(gdf, source, layer_type):
    cols = ["geometry", "area_ha"]
    for c in ["nome", "name", "descrizione", "NOME", "NAME"]:
        if c in gdf.columns:
            cols.append(c)
            break
    return gdf[cols].assign(source=source, layer_type=layer_type).copy()


# ── Edit this dict to match what you saw in the comparison above ─────────────
# Keys become layer names inside the .gpkg.
# The isochrone notebook reads layers named "osm_public" and/or "municipality".
layers_to_save = {
    "osm_public": prep_save(osm_public, "OSM", "osm_public"),
}
if muni_raw is not None:
    layers_to_save["municipality"] = prep_save(muni_raw, "MUNICIPALITY", "municipality")
# Uncomment to also include pe_unis DBSN layer:
# if pe_green is not None:
#     layers_to_save["pe_unis"] = prep_save(pe_green, "DBSN", "pe_unis")
# ─────────────────────────────────────────────────────────────────────────────

import os as _os
_os.makedirs(_os.path.dirname(OUT_GPKG), exist_ok=True)

for layer_name, gdf in layers_to_save.items():
    gdf.to_file(OUT_GPKG, layer=layer_name, driver="GPKG")
    print(f"  saved {len(gdf):,} features → {OUT_GPKG} [{layer_name}]")